In [ ]:
# Day 15 - Chatbot with Conversation Memory (Fixed Working Version)

!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers

# Correct Imports
from langchain_huggingface import HuggingFacePipeline
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain_core.prompts import PromptTemplate

import torch
from transformers import pipeline

print("✅ All packages installed and imported successfully!")

In [ ]:
# Load the LLM
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=300,
    temperature=0.7,
    pad_token_id=50256
)

llm = HuggingFacePipeline(pipeline=pipe)
print("✅ LLM Loaded!")

In [ ]:
# Create Memory + Conversation Chain
memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True   # Set to False if you don't want to see the internal prompt
)

print("✅ Chatbot with Memory is Ready!")

In [ ]:
# Test Multi-turn Conversation
print("=== Starting Conversation ===\n")

response1 = conversation.predict(input="Hi, my name is Hayat.")
print("AI:", response1, "\n")

response2 = conversation.predict(input="What is my name?")
print("AI:", response2, "\n")

response3 = conversation.predict(input="Tell me something interesting about Ethiopia.")
print("AI:", response3, "\n")

response4 = conversation.predict(input="What did I tell you my name is?")
print("AI:", response4, "\n")

In [ ]:

custom_prompt = PromptTemplate(
    input_variables=["history", "input"],
    template="""You are a friendly Ethiopian AI assistant.

Conversation History:
{history}

User: {input}
Assistant:"""
)

conversation.prompt = custom_prompt